### Tools
Models Can request to call tools that perform tasks such as fetching data from a database, searching the web or running code. Tools are fearing of : 
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON Schema) 
2. A function or coroutine to execute

In [6]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("openai/gpt-oss-120b", model_provider="groq")

# response = model.invoke("why do I like girl?")
# response.content

In [ ]:
from langchain.tools import tool
import requests

@tool
def getWeather(location: str) -> str:
    """Get current weather information for a city."""
    if "kolkata" in location.lower():
        return "The weather in Kolkata is currently rainy."
    return "sunny"

model_with_tools = model.bind_tools([getWeather])

In [8]:
response = model_with_tools.invoke("What's the weather in Dubai?")
print(response)
for tool_call in response.tool_calls:
    # tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Tool: {tool_call['args']}")

content='' additional_kwargs={'reasoning_content': 'User asks for weather in Dubai. Use getWeather function.', 'tool_calls': [{'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'function': {'arguments': '{"location":"Dubai"}', 'name': 'getWeather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 127, 'total_tokens': 167, 'completion_time': 0.083790597, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.004881125, 'prompt_tokens_details': None, 'queue_time': 0.402363824, 'total_time': 0.088671722}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_90620edd96', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0baab-671d-70c1-813e-9def87b7e1fa-0' tool_calls=[{'name': 'getWeather', 'args': {'location': 'Dubai'}, 'id': 'fc_f6856439-237c-48a1-bb9f-7be075778a46', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 12

In [1]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    if "kolkata" in location.lower():
        return "The weather in Kolkata is currently rainy."
    return "sunny"

# Create the agent with the weather tool
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[get_weather]
)


In [3]:
response = agent.invoke({
    "messages": [("user", "What's the weather in Dubai?")]
})
print(response)
for msg in response["messages"]:
    tool_calls = getattr(msg, "tool_calls", None)
    for tool_call in tool_calls:
        print(f"Tool: {tool_call['name']}")
        print(f"Args: {tool_call['args']}")

{'messages': [HumanMessage(content="What's the weather in Dubai?", additional_kwargs={}, response_metadata={}, id='a5223fc9-9229-41cc-876e-528df26be69f'), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "Dubai"}'}, '__gemini_function_call_thought_signatures__': {'call_1180251': 'EnEKbwFpFH0T/aQtVTIbQKlmqh4BuDXOc2Xy/iTY5bby6QMQl5Q3OcGuuxixiLFwCfYTbMZMFzN84VmRw61m+hOxHadDLALmNM4lr7VvGUBAalCF4ZOGA+qzH3J6ByY3zv257T2kEvtCLx/OssiGy+KdsA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0bcd1-8ee4-7600-9351-c0f9a5712ee2-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Dubai'}, 'id': 'call_1180251', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 55, 'output_tokens': 16, 'total_tokens': 71, 'input_token_details': {'cache_read': 0}}), ToolMessage(content='sunny', name='get_weather', 

TypeError: 'NoneType' object is not iterable